In [23]:
import pandas as pd
from sklearn.preprocessing import OrdinalEncoder
from sklearn.cluster import KMeans
from sklearn.preprocessing import MinMaxScaler
from sklearn import tree
import numpy as np
import itertools

In [24]:
ruta = '../Data/'
clientes = pd.read_excel(f'{ruta}Clientes.xlsx')
Detalle_ventas = pd.read_excel(f'{ruta}Detalle_ventas.xlsx')
Productos = pd.read_excel(f'{ruta}Productos.xlsx')
Ventas = pd.read_excel(f'{ruta}Ventas.xlsx')

In [25]:
enc2 = OrdinalEncoder()

df_pronostico = (
    Ventas[['fecha', 'id_venta']].merge(Detalle_ventas[['id_venta', 'cantidad', 'nombre_producto']], on = 'id_venta')
)

df_pronostico['nombre_producto'] = enc2.fit_transform(df_pronostico[['nombre_producto']])

df_pronostico = (
    pd.DataFrame(df_pronostico.sort_values(by = 'fecha')
                 .groupby(by = ['fecha', 'nombre_producto'])['cantidad'].sum())
                 .reset_index()
                 )

fechas = pd.date_range(start=df_pronostico["fecha"].min(), end=df_pronostico["fecha"].max())
productos = df_pronostico['nombre_producto']
multi_index = pd.MultiIndex.from_product([fechas, productos], names=["fecha", 'nombre_producto'])
df_pronostico = df_pronostico.set_index(["fecha", 'nombre_producto']).reindex(multi_index, fill_value=0).reset_index()

df_pronostico["anio"] = df_pronostico["fecha"].dt.year
df_pronostico["mes"] = df_pronostico["fecha"].dt.month
df_pronostico["dia"] = df_pronostico["fecha"].dt.day

porcentaje = 0.8
X = df_pronostico[['anio', 'mes', 'dia', 'nombre_producto']]
y = df_pronostico[['cantidad']]
X_train, X_test = X[:int(round(len(df_pronostico)*porcentaje,0))], X[int(round(len(df_pronostico)*(1-porcentaje))):]
y_train, y_test = y[:int(round(len(df_pronostico)*porcentaje,0))], y[int(round(len(df_pronostico)*(1-porcentaje))):]
modelo = tree.DecisionTreeRegressor()
modelo.fit(X_train, y_train)

DecisionTreeRegressor()

In [26]:
datos_pronosticos = pd.DataFrame(
    list(itertools.product([2024], [7], range(1, 15), range(0, 95))),
    columns=["anio", "mes", "dia", "nombre_producto"]
)
Cantidad_pronosticada = modelo.predict(datos_pronosticos)

In [27]:
datos_pronosticos['fecha'] = pd.to_datetime(
    datos_pronosticos.rename(
        columns={'anio': 'year', 'mes': 'month', 'dia': 'day'}
    )[['year', 'month', 'day']]
)
datos_pronosticos['nombre_producto'] = np.array(enc2.inverse_transform(datos_pronosticos[['nombre_producto']])).flatten()
datos_pronosticos['id_venta'] = [i for i in range(121, len(datos_pronosticos) + 121)]
datos_pronosticos['cantidad'] = Cantidad_pronosticada
datos_pronosticos = datos_pronosticos.merge(Productos[['id_producto', 'nombre_producto']], on = 'nombre_producto')
datos_pronosticos['id_cliente'] = 101
datos_pronosticos['medio_pago'] = 'Pronostico'

In [28]:
df_cluster = (
    Ventas[['id_cliente', 'medio_pago', 'id_venta']]
    .merge(Detalle_ventas[['id_venta', 'cantidad', 'importe']], on = 'id_venta')
)
enc = OrdinalEncoder()
df_cluster['medio_pago'] = enc.fit_transform(df_cluster[['medio_pago']])
df_cluster = (
    pd.DataFrame(df_cluster.groupby("id_cliente")['medio_pago'].agg(lambda x: x.mode().iat[0]))
    .merge(
        df_cluster.groupby("id_cliente")[['cantidad', 'importe']].sum(),
        on='id_cliente'
    )
)
X = df_cluster[['medio_pago', 'cantidad', 'importe']]
min_max_scaler = MinMaxScaler()
X_train_minmax = min_max_scaler.fit_transform(X)
kmeans = KMeans(n_clusters=3, random_state=0, n_init="auto").fit(X_train_minmax)
kmeans.cluster_centers_
X["cluster"] = kmeans.labels_

# Agregar modelos a los datos

In [29]:
ruta_guardar = './Data/'

In [30]:
clientes = (clientes.merge(X.reset_index()[['id_cliente', 'cluster']],
                on = 'id_cliente')
)
clientes['cluster'] = (clientes['cluster']
                    .replace({0:"Clientes Oro",
                              1:"Clientes Digitales",
                              2:"Clientes Express"})
                        )
clientes.loc[len(clientes)] = [
    101, 'Pronostico', 'Pronostico', 'Pronostico', pd.Timestamp('2024-07-15'), 'Pronostico'
]
clientes.to_excel(f'{ruta_guardar}clientes.xlsx', index=False)

In [31]:
Detalle_ventas = Detalle_ventas[['id_venta', 'id_producto', 'cantidad']]
Detalle_ventas = (
    pd.concat([Detalle_ventas, datos_pronosticos[Detalle_ventas.columns]])
)
Detalle_ventas = Detalle_ventas[Detalle_ventas['cantidad']>0]
Detalle_ventas.to_excel(f'{ruta_guardar}Detalle_ventas.xlsx', index=False)

In [32]:
Ventas = Ventas[['id_venta', 'fecha', 'id_cliente', 'medio_pago']]
Ventas = (
    pd.concat([Ventas, datos_pronosticos[Ventas.columns]])
)
Ventas.to_excel(f'{ruta_guardar}Ventas.xlsx', index=False)

In [33]:
Productos

,id_producto,nombre_producto,categoria,precio_unitario
0,1,Coca Cola 1.5L,Alimentos,2347
1,3,Sprite 1.5L,Alimentos,4964
2,5,Agua Mineral 500ml,Alimentos,4777
3,7,Jugo de Manzana 1L,Alimentos,3269
4,9,Yerba Mate Suave 1kg,Alimentos,3878
...,...,...,...,...
95,92,Crema Dental 90g,Cuidado personal,2512
96,94,Hilo Dental,Cuidado personal,1418
97,96,Suavizante 1L,Limpieza,4920
98,98,Desengrasante 500ml,Limpieza,2843
